In [ ]:
# COLAB SETUP — run this cell first every session
# WHY: Google Colab resets its filesystem and Python environment on every
# runtime restart. pip install pyspark bundles Spark binaries inside the
# Python package so no SPARK_HOME or local Spark installation is needed.
# The original os.environ cell hardcoded a local Mac path that does not
# exist on Colab's VM and has been removed from all notebooks.
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("04-RDD-Operations") \
    .getOrCreate()

sc = spark.sparkContext
print(spark.version)

In [ ]:
# DATA PATH SETUP
# WHY: If running this notebook without first running 01-PySpark-Get-Started,
# this cell clones the repo data and downloads the NYC TLC dataset.
# It is idempotent — safe to run multiple times.
import os, shutil, subprocess

os.makedirs('/content/data', exist_ok=True)

# Clone repo data files if not present
if not os.path.exists('/content/pyspark-tutorial'):
    subprocess.run(['git','clone','https://github.com/coder2j/pyspark-tutorial.git',
                    '/content/pyspark-tutorial'], check=True)
shutil.copytree('/content/pyspark-tutorial/data', '/content/data', dirs_exist_ok=True)

# Download NYC TLC dataset if not present
if not os.path.exists('/content/data/trips.parquet'):
    os.system('wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet -O /content/data/trips.parquet')
    trips = spark.read.parquet('/content/data/trips.parquet')
    trips.limit(10000).write.mode('overwrite').option('header',True).csv('/content/data/trips_csv')
    trips.limit(1000).write.mode('overwrite').json('/content/data/trips_json')

print('Data ready:', os.listdir('/content/data'))

In [3]:
from pyspark.sql import SparkSession

In [4]:
# Create a SparkSession
spark = SparkSession.builder.appName("RDD-Demo").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/07/16 18:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### How to create RDDs

In [5]:
numbers = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(numbers)

In [6]:
# Collect action: Retrieve all elements of the RDD
rdd.collect()

[1, 2, 3, 4, 5]

In [7]:
# Create an RDD from a list of tuples
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35), ("Alice", 40)]
rdd = spark.sparkContext.parallelize(data)

In [8]:
# Collect action: Retrieve all elements of the RDD
print("All elements of the rdd: ", rdd.collect())

All elements of the rdd:  [('Alice', 25), ('Bob', 30), ('Charlie', 35), ('Alice', 40)]


### RDDs Operation: Actions 

In [9]:
# Count action: Count the number of elements in the RDD
count = rdd.count()
print("The total number of elements in rdd: ", count)

The total number of elements in rdd:  4


In [10]:
# First action: Retrieve the first element of the RDD
first_element = rdd.first()
print("The first element of the rdd: ", first_element)

The first element of the rdd:  ('Alice', 25)


In [11]:
# Take action: Retrieve the n elements of the RDD
taken_elements = rdd.take(2)
print("The first two elements of the rdd: ", taken_elements)

The first two elements of the rdd:  [('Alice', 25), ('Bob', 30)]


In [12]:
# Foreach action: Print each element of the RDD
rdd.foreach(lambda x: print(x))

('Charlie', 35)
('Alice', 25)
('Bob', 30)
('Alice', 40)


### RDDs Operation: Transformations 

In [13]:
# Map transformation: Convert name to uppercase
mapped_rdd = rdd.map(lambda x: (x[0].upper(), x[1]))

In [14]:
result = mapped_rdd.collect()
print("rdd with uppercease name: ", result)

rdd with uppercease name:  [('ALICE', 25), ('BOB', 30), ('CHARLIE', 35), ('ALICE', 40)]


In [15]:
# Filter transformation: Filter records where age is greater than 30
filtered_rdd = rdd.filter(lambda x: x[1] > 30)
filtered_rdd.collect()

[('Charlie', 35), ('Alice', 40)]

In [16]:
# ReduceByKey transformation: Calculate the total age for each name
reduced_rdd = rdd.reduceByKey(lambda x, y: x + y)
reduced_rdd.collect()

[('Alice', 65), ('Bob', 30), ('Charlie', 35)]

In [17]:
# SortBy transformation: Sort the RDD by age in descending order
sorted_rdd = rdd.sortBy(lambda x: x[1], ascending=False)
sorted_rdd.collect()

[('Alice', 40), ('Charlie', 35), ('Bob', 30), ('Alice', 25)]

### Save RDDs to text file and read RDDs from text file

In [18]:
# Save action: Save the RDD to a text file
rdd.saveAsTextFile("output.txt")

In [19]:
# create rdd from text file
rdd_text = spark.sparkContext.textFile("output.txt")
rdd_text.collect()

["('Alice', 40)", "('Bob', 30)", "('Alice', 25)", "('Charlie', 35)"]

### Shut down Spark Session

In [20]:
spark.stop()